# PHA 领域专家 Agent 示例

本 Notebook 展示 PHA 的**领域专家 Agent**。它通过 OneTwo 使用 **ReAct 框架**，逐步推理健康问题，借助网页搜索、Data Commons、代码执行和参考范围等工具解读结果。

## 环境要求

安装 OneTwo 可启用完整的 ReAct 功能。

```bash
pip install git+https://github.com/google-deepmind/onetwo
```

缺少 OneTwo 时，Agent 会以简化模式运行。

## 环境搭建

In [ ]:
# 安装依赖关系( 需要时不做评论)
# !pip install git+https://github.com/google-deepmind/onetwo  # For ReAct support
# !pip install google-genai openai # LLM backends
# !pip install tavily-python  # Better search (optional)
# !pip install duckduckgo-search  # Free search fallback

In [ ]:
import os
import sys

# --- Configuration ---
# 在此设定您的 API 密钥和提供者
API_KEY = "your-api-key-here"
# 提供方选项："基因","openai","人类"
PROVIDER = "gemini"

# 可选：Tavily API 键 领域专家 Agent (搜索).
TAVILY_API_KEY = "your-tavily-api-key-here"


In [ ]:
# 检查是否有1 2 个可用
from pha.agents import is_react_available

print(f"OneTwo ReAct available: {is_react_available()}")
if not is_react_available():
    print("Note: Running in fallback mode without full ReAct support.")
    print("For full functionality, install: pip install git+https://github.com/google-deepmind/onetwo")

## 初始化领域专家 Agent

In [ ]:
from pha.agents import DomainExpertAgent

# 创建代理
# 默认使用Tavily进行高质量搜索(需要TAVILY_API_KEY).
# 设定 search_backend ='duckduckgo' 用于免费但质量较低的搜索
agent = DomainExpertAgent(
    search_backend='tavily',  # 或"Duckduckgo"如果你没有 Tavily 键
    tavily_api_key=TAVILY_API_KEY,
)

# 配置设置( 使用全局 LLM 配置)
import glob

# 查找实例笔记本( PHIA 模式)
exemplar_files = glob.glob('../few_shots/*.ipynb')
print(f"Found {len(exemplar_files)} exemplar notebooks")

agent.get_agent(
    api_key=API_KEY,
    provider=PROVIDER,
    exemplar_files=exemplar_files,
)

print("Domain Expert Agent initialized!")
print(f"Using ReAct: {is_react_available()}")


## 提供用户健康数据（可选）

Agent 可以结合用户健康数据给出个性化回答。

In [ ]:
# 手动设定用户健康数据
agent.set_user_health_data("""
## User Profile
- Age: 45
- Sex: Male
- Height: 178 cm
- Weight: 82 kg
- BMI: 25.9

## Recent Lab Results
- LDL Cholesterol: 142 mg/dL
- HDL Cholesterol: 52 mg/dL
- Total Cholesterol: 210 mg/dL
- Triglycerides: 145 mg/dL
- Fasting Glucose: 98 mg/dL
- HbA1c: 5.7%

## Wearable Data (7-day average)
- Daily Steps: 7,500
- Resting Heart Rate: 68 bpm
- Heart Rate Variability: 42 ms
- Sleep Duration: 6.5 hours
""")

print("User health data set!")

## 提出健康问题

Agent 会按 ReAct 流程处理。

1. 分析问题。
2. 调用工具，如搜索、计算或查询参考范围。
3. 查看返回结果。
4. 循环执行，直到得到回答。

In [ ]:
# 示例1:解释实验室结果。
question = "Based on my lab results, should I be concerned about my cholesterol levels?"

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

In [ ]:
# 示例 2:询问具体衡量标准
question = "What does my HbA1c of 5.7% indicate about my diabetes risk?"

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

In [ ]:
# 示例3:询问可穿戴的度量衡
question = "How does my resting heart rate and HRV compare to healthy norms for my age?"

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

In [ ]:
# 实例4:全面健康评估
question = """Given all my health data, what are the top 3 things I should focus on 
to improve my overall health? Please provide specific, actionable recommendations."""

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

## 从样本文件加载健康数据

In [ ]:
# 装入样本数据
from config import Settings
from pha.utils import load_persona

settings = Settings(data_dir='../data/sample')
summary_df, activities_df, profile_df, population_df = load_persona(settings=settings)

# 格式化代理商的最近数据
recent_7_days = summary_df.tail(7)

health_summary = f"""
## User Profile
- Age: {profile_df['age'].iloc[0]}
- Sex: {profile_df['sex'].iloc[0]}
- Height: {profile_df['height_cm'].iloc[0]} cm
- Weight: {profile_df['weight_kg'].iloc[0]} kg

## 7-Day Wearable Summary (averages)
- Daily Steps: {recent_7_days['steps'].mean():.0f}
- Sleep Duration: {recent_7_days['sleep_minutes'].mean():.0f} minutes
- Resting Heart Rate: {recent_7_days['resting_heart_rate'].mean():.1f} bpm
- Heart Rate Variability: {recent_7_days['heart_rate_variability'].mean():.1f} ms
- Sleep Score: {recent_7_days['sleep_score'].mean():.1f}
"""

agent.set_user_health_data(health_summary)
print("Loaded sample health data:")
print(health_summary)

In [ ]:
# 询问装入的数据
question = "Based on my wearable data, how is my sleep quality and what can I do to improve it?"

print(f"Question: {question}")
print("\nAgent reasoning...")

response = agent.call_agent(question)
print(f"\nResponse:\n{response}")

## 自己试着提问

In [ ]:
# 问你自己的健康问题
your_question = "What lifestyle changes would have the biggest impact on my cardiovascular health?"

response = agent.call_agent(your_question)
print(response)